In [5]:
!pip install transformers accelerate sentencepiece matplotlib seaborn

In [15]:
import torch

from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM

model_name = "Qwen/Qwen3-1.7B"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto",
    attn_implementation="eager"
)

model.eval()

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2048)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2048, out_features=6144, bias=False)
          (up_proj): Linear(in_features=2048, out_features=6144, bias=False)
          (down_proj): Linear(in_features=6144, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((2048,), eps=1e-06)
        (post_attention_layer

In [3]:
!git clone https://github.com/hkchi-pham/attention-sink-research.git

Cloning into 'attention-sink-research'...
remote: Enumerating objects: 327, done.
remote: Counting objects: 100% (93/93), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 327 (delta 55), reused 5 (delta 5), pack-reused 234 (from 1)
Receiving objects: 100% (327/327), 477.75 KiB | 2.45 MiB/s, done.
Resolving deltas: 100% (84/84), done.


In [7]:
import json

with open(
    "/content/attention-sink-research/docs/research-logs/phase_1/prompt/english.json",
    "r",
    encoding="utf-8"
) as f:

    english = json.load(f)

with open(
    "/content/attention-sink-research/docs/research-logs/phase_1/prompt/vietnamese.json",
    "r",
    encoding="utf-8"
) as f:

    vietnamese = json.load(f)

In [11]:
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

save_dir = Path("/content/drive/MyDrive/attention-sink/data/attentions")
save_dir.mkdir(parents=True, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
print(model.config._attn_implementation)

import transformers
print(transformers.__version__)

import torch
print(torch.__version__)

eager
5.12.0
2.11.0+cu128


In [22]:
with torch.no_grad():
    outputs = model(
        **inputs,
        output_attentions=True,
        return_dict=True
    )

print(outputs.keys())


print(len(outputs.attentions))

odict_keys(['logits', 'past_key_values', 'attentions'])
28


In [23]:
datasets = {
    "english": english,
    "vietnamese": vietnamese
}

import gc

for language, dataset in datasets.items():
  for category in dataset:
    for prompt_idx, prompt in enumerate(dataset[category]):
      inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
      tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
      token_positions = list(enumerate(tokens))
      with torch.no_grad():
        outputs = model( **inputs, output_attentions=True)

      attentions = outputs.attentions
      save_data = {
              "language": language,
              "category": category,
              "prompt": prompt,
              "tokens": tokens,
              "token_positions": token_positions,
              "seq_len": len(tokens),
              "input_ids": inputs["input_ids"].cpu(),
              "attentions": [layer.cpu() for layer in attentions]
              }
      filename = (
              f"{language}_"
              f"{category}_"
              f"{prompt_idx+1:02d}.pt"
              )

      torch.save(save_data,save_dir / filename)
      print(f"{filename} | layers = {len(attentions)}")

      print(f"[{language}] [{category}] Prompt {prompt_idx+1} saved.")

      del outputs
      gc.collect()
      if torch.cuda.is_available():
        torch.cuda.empty_cache()





english_question_01.pt | layers = 28
[english] [question] Prompt 1 saved.
english_question_02.pt | layers = 28
[english] [question] Prompt 2 saved.
english_question_03.pt | layers = 28
[english] [question] Prompt 3 saved.
english_question_04.pt | layers = 28
[english] [question] Prompt 4 saved.
english_question_05.pt | layers = 28
[english] [question] Prompt 5 saved.
english_question_06.pt | layers = 28
[english] [question] Prompt 6 saved.
english_question_07.pt | layers = 28
[english] [question] Prompt 7 saved.
english_question_08.pt | layers = 28
[english] [question] Prompt 8 saved.
english_question_09.pt | layers = 28
[english] [question] Prompt 9 saved.
english_question_10.pt | layers = 28
[english] [question] Prompt 10 saved.
english_story_01.pt | layers = 28
[english] [story] Prompt 1 saved.
english_story_02.pt | layers = 28
[english] [story] Prompt 2 saved.
english_story_03.pt | layers = 28
[english] [story] Prompt 3 saved.
english_story_04.pt | layers = 28
[english] [story] Pro

In [13]:
from pathlib import Path

drive_dir = Path("/content/drive/MyDrive/attention-sink/data/attentions")
drive_dir.mkdir(parents=True, exist_ok=True)

import shutil

local_dir = Path("data/attentions")

for file in local_dir.glob("*.pt"):
    shutil.copy(file, drive_dir / file.name)

print("Done!")

Done!
